# LocalGPT — a chat loop against a model running on your own machine

### 521 · Session 1 lab (part 2 of 2)

This is the **minimal** counterpart to `tavily_weather_agent.ipynb`. That notebook builds an *agent* — a model with a tool it can decide to call. This one strips all of that away and leaves the bare thing underneath: **a loop that sends text to a model and prints what comes back.**

Run this one **first**. It answers one question — *is my local model actually working?* — and if the answer is no, nothing in the agent notebook will run either.

---

### Why local, and why it matters for this course

| | Cloud API (OpenAI, Anthropic) | Local (Ollama) |
|---|---|---|
| **Cost** | Per token, forever | Free after download |
| **Latency** | Network round-trip | Your GPU/CPU |
| **Privacy** | Prompt leaves your machine | Never leaves |
| **Quality** | Frontier models | Smaller, weaker |
| **Availability** | Needs internet + key | Works on a plane |

Session 3 covers this trade-off formally as **the model landscape**. Doing it by hand now means that session is a recap, not a first encounter.

### What to notice while it runs

1. **The first token takes noticeably longer than the rest.** That's *prefill* — the model reading your whole prompt at once. Everything after is *decode*, one token at a time. 536 session 5 is entirely about this asymmetry.
2. **This loop has no memory.** Ask "what's the capital of France?", then "what's its population?" — it won't know what *its* means. Each `generate()` call is independent. Adding memory is 521 session 6; you're seeing the problem it solves.
3. **It's `generate`, not `chat`.** No system prompt, no role structure — raw next-token prediction, which is 536's section 3 with a `while` loop around it.

## Step 0 — Before you run anything

1. **Install Ollama** — https://ollama.com/download (macOS, Windows, Linux)
2. **Pull a model** in your terminal:
   ```bash
   ollama pull llama3          # ~4.7 GB, needs ~8 GB RAM
   # or, on a lighter machine:
   ollama pull gemma:2b        # ~1.7 GB
   ```
3. **Install the Python client:**
   ```bash
   pip install ollama
   ```
4. **Leave Ollama running** in the background — the Python library talks to it over `http://localhost:11434`. If it isn't running you'll get `Connection refused`.

**Disk-space warning:** models are large and they accumulate. `ollama list` shows what you have; `ollama rm <model>` removes one.

## Step 1 — Confirm the model is downloaded and the server is up

`ollama list` prints every model on your machine with its size and digest.

**Reading the output:** if you see a table with `llama3` (or whatever you pulled) in it, you're ready. If you get `command not found`, Ollama isn't installed or isn't on your `PATH`. If you get a connection error, the app isn't running.

⚠️ **Whatever name appears in this table is the name you must use in the next cell.** `llama3` and `llama3.2` are different models; `gemma:2b` needs the tag. Copy the name exactly.

In [1]:
! ollama list


]11;?\NAME                       ID              SIZE      MODIFIED     
qwen2.5:3b                 357c53fb659c    1.9 GB    4 days ago      
qwen2.5:3b-instruct        357c53fb659c    1.9 GB    11 days ago     
llama3.2:1b                baf6a787fdff    1.3 GB    2 weeks ago     
nomic-embed-text:latest    0a109f422b47    274 MB    3 weeks ago     
qwen2:latest               dd314f039b9d    4.4 GB    3 weeks ago     
qwen2.5:0.5b               a8b0c5157701    397 MB    3 weeks ago     
qwen2.5vl:7b               5ced39dfa4ba    6.0 GB    3 weeks ago     
qwen:latest                d53d04290064    2.3 GB    3 weeks ago     
qwen3:latest               500a1f067a9f    5.2 GB    3 weeks ago     
mistral:latest             6577803aa9a0    4.4 GB    2 months ago    
llama3.2:3b                a80c4f17acd5    2.0 GB    3 months ago    
gemma3:1b                  8648f39daa8f    815 MB    5 months ago    
phi3:mini                  4f2222927938    2.2 GB    5 months ago    
phi3:latest    

## Step 2 — The chat loop

Eleven lines, and every one earns its place:

| Line | What it does | Course concept |
|---|---|---|
| `while True:` | Keeps the conversation open | The dialogue loop — 521 section 4 |
| `input(...)` | Reads your turn | User request — lifecycle stage 1 |
| `if ... in ['exit','quit']` | An exit that isn't `Ctrl-C` | — |
| `ollama.generate(model=, prompt=)` | Sends the prompt, waits for the full answer | Next-token prediction — 536 section 3 |
| `response['response']` | Pulls the text out of the reply dict | — |

⚠️ **Change `model=` to match your `ollama list` output** before running, or you'll get a "model not found" error.

**How to stop it:** type `exit` or `quit`. If the loop hangs, interrupt the kernel (■ button, or `Ctrl-C` / `I,I`).

---

### 🔬 Three things to try — this is the actual lab

Reading this notebook teaches nothing. Changing it teaches the session.

1. **Break its memory.** Ask two questions where the second depends on the first. Watch it fail. That failure is the entire motivation for session 6.
2. **Add `stream=True`.**
   ```python
   for chunk in ollama.generate(model='llama3', prompt=question, stream=True):
       print(chunk['response'], end='', flush=True)
   ```
   Tokens appear one at a time instead of all at once. **You are literally watching autoregressive decoding** — the mechanism from 536 section 3 made visible. Nothing about the model changed; only when you're shown the output.
3. **Give it a system prompt.**
   ```python
   ollama.generate(model='llama3', prompt=question,
                   system="You are a terse assistant. Answer in one sentence.")
   ```
   Same model, same question, different behaviour — with no retraining. That gap between *capability* and *elicited behaviour* is what prompt engineering is (536 section 10), and why it's a skill rather than a trick.

**Then compare.** Run this notebook, then the Tavily one. Ask both *"what's the weather in Tokyo right now?"* This one will confidently make something up — a hallucination from training data with no way to check. The agent will go and look. That contrast, run yourself, is the clearest argument for tools you'll get all semester.

In [2]:
import ollama

while True:
    question = input("Ask something (or type 'exit' to quit): ")

    if question.lower() in ['exit', 'quit']:
        print("Goodbye! 👋")
        break

    response = ollama.generate(model='gemma3:1b', prompt=question)

    print("Answer:", response['response'])

Answer: Hey there! How can I help you today? 😄 What's up?
Answer: Today is Friday, July 10th, 2024 at 3:56 PM PST.

To check more localized times, you can use these websites:**

*   **Time and Date: [https://www.timeanddate.com/](https://www.timeanddate.com/)**
*   **World Time Server: [https://www.worldtimereference.org/](https://www.worldtimереreference.org/)**
Answer: Currently, as of today (June 7, 2024), **Argentina** has the most FIFA World Cup titles. They have a total of  লক্ষ্য.

Here's the history:

1.  **Uruguay:** 2 (1930, 1950)
2.  **Italy:** 4 (1934, 1998, 2006, 2018)
3.  **Brasil:** 7 (1950, 1990, 2002, 2014, 2017, 2019, 2022) 


Goodbye! 👋


## Troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `ConnectionError` / `Connection refused` | Ollama server isn't running | Launch the Ollama app; check http://localhost:11434 in a browser |
| `model 'X' not found` | Name doesn't match a pulled model | Run `ollama list`, copy the exact name including any `:tag` |
| Very slow, machine fans spin up | Model too large for your RAM, swapping to disk | Pull a smaller one: `ollama pull gemma:2b` |
| `ModuleNotFoundError: ollama` | Python client not installed | `pip install ollama` |
| Loop won't stop | `input()` is blocking | Interrupt the kernel, or type `exit` |
| Answers are odd or repetitive | Small model, no sampling controls | Try a larger model, or pass `options={'temperature': 0.7}` |

## What this connects to

| Here | Session note | Where it's taught properly |
|---|---|---|
| Local vs cloud trade-off | section 5 Frameworks | 521 S3 — model landscape & cost |
| No memory between turns | section 4 The six components | 521 S6 — agent memory |
| Token-by-token generation | section 6 Tokenization | 536 S1 section 3, 536 S5 — inference |
| System prompt changes behaviour | section 8 LLMs as the brain | 536 S10 — prompt engineering |
| It confabulates the weather | section 8, failure modes | 521 S7 — RAG · S4 — tools |